# Equity underlying

Everything the market holds on one equity, through the `tq.Equity` view: market keys and risk factors, spot, repo,
dividends, vol surface, correlations; then the forward, the implied dividend yield and their AAD sensitivities.
Docs: *Architecture* (The Equity view), *Models and Pricing* (Underlying, forward and dividends), *Exercises* (`equity.tex`).

In [ ]:
import os
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq

## The Equity view
`mkt.get_equity(ticker)`: references (not copies) to spot, repo, dividend yield or dividends, vol surface and the
discount curve of the currency. Dividend model from the data: `DIV` present → discrete, otherwise `DIVYIELD`.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date

TICKER = "TICKER1"
eq = mkt.get_equity(TICKER)
expiry = EVALUATION_DATE + timedelta(days=365)
print(eq)
eq.summary(expiry)

In [ ]:
print("same variables as the market:", eq.spot is mkt.get_eq_spot(TICKER), eq.vol_surface is mkt.get_eq_vol_surface(TICKER))

## Market keys and risk factors
`eq.risk_factors()`: the market variables this underlying depends on (a subset of `mkt.risk_factors()`).

In [ ]:
prefix = f"EQ:{eq.currency.name}:{TICKER}:"
print("market keys:", [k for k in mkt.risk_factors() if k.startswith(prefix)])
pd.DataFrame([{"key": k, "variables": len(v), "shape": [tuple(x.shape) for x in v][0] if len(v) == 1 else f"{len(v)} x ()",
               "dtype": v[0].dtype.name} for k, v in eq.risk_factors().items()])

## Dividends
Cash amounts with ex-date and payment date (the amounts are one variable: one sensitivity per dividend). A dividend
belongs to a horizon by its **ex-date** (the stock drops then); `pv_dividends` discounts it from its **payment date**
(14 days later here).

In [ ]:
div = eq.dividends.to_dataframe()
curve, spot = eq.discount_curve, float(eq.spot)
div["% spot"] = 100 * div["amount"] / spot
div["P(0, pay)"] = [float(curve.discount(d)) for d in div["payment_date"]]
div["PV"] = div["amount"] * div["P(0, pay)"]
print(f"PV of the dividends up to {expiry}: {float(eq.pv_dividends(expiry)):.4f}  "
      f"(= sum of 'PV' with ex_date <= expiry)")
div.round(5)

## Volatility surface
Black vols on (maturity × absolute strike) nodes; below in K / spot. The ATM vol of a maturity is the vol at the
forward $K = F(T)$, not at the spot (dividends and rates move the forward away from $S_0$).

In [ ]:
vs = eq.vol_surface
pd.DataFrame(100 * vs.volatility_matrix.numpy(), index=[f"T={t:.2f}" for t in vs.maturity],
             columns=[f"{k / spot:.2f}" for k in vs.strike]).round(2)

In [ ]:
atm = pd.DataFrame({"T": vs.maturity})
atm["forward"] = [float(eq.forward(t)) for t in vs.maturity]
atm["F / S"] = atm["forward"] / spot
atm["vol(K=S) %"] = [100 * float(vs.volatility(strike=spot, tenor=t)) for t in vs.maturity]
atm["vol(K=F) %"] = [100 * float(vs.volatility(strike=f, tenor=t)) for f, t in zip(atm["forward"], vs.maturity)]
atm["total var (K=F)"] = (atm["vol(K=F) %"] / 100) ** 2 * atm["T"]
atm.round(4)

## Correlations
One matrix for all the factors (`CORR:ALL:GLOBAL:SPOT`), names = market-key prefixes; one variable (gradient on each pair).

In [ ]:
corr = mkt.get_correlation_matrix()
name = f"EQ:{eq.currency.name}:{TICKER}"
row = pd.Series(corr.variables[0].numpy()[corr.names.index(name)], index=corr.names).drop(name)
print(f"{len(corr.names)} factors; correlations of {name}: min {row.min():.2f}, mean {row.mean():.2f}, max {row.max():.2f}")
pd.concat([row.nlargest(3), row.nsmallest(3)]).round(3).to_frame("corr")

## Forward term structure
Discrete dividends: $F(T) = (S_0 - PV_{div}(T))\,e^{-q_{repo}T}/P(0,T)$, $PV_{div}(T) = \sum_{ex_i \le T} D_i\,P(0,pay_i)$,
dropping at each ex-date.

In [ ]:
days = np.arange(1, 2 * 365 + 1, 3)
dates = [EVALUATION_DATE + timedelta(days=int(d)) for d in days]
T = days / 365
fwd = [float(eq.forward(d)) for d in dates]
no_div = [spot * np.exp(-float(eq.repo) * t) / float(curve.discount(d)) for t, d in zip(T, dates)]

plt.figure(figsize=(7, 4))
plt.plot(T, no_div, "--", label="no dividends")
plt.plot(T, fwd, label="with discrete dividends")
for ex in eq.dividends.ex_dates:
    plt.axvline((ex - EVALUATION_DATE).days / 365, color="grey", lw=0.8, ls=":")
plt.xlabel("T (years)"); plt.ylabel("forward"); plt.title(f"{TICKER} forward"); plt.grid(alpha=0.3); plt.legend();

## Implied dividend yield
$q_{eff}(T) = -\ln(1 - PV_{div}(T)/S_0)/T$: the continuous yield giving the same forward; steps at the ex-dates, then decays.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(T, [100 * float(eq.implied_div_yield(d)) for d in dates])
plt.xlabel("T (years)"); plt.ylabel("q_eff (%)"); plt.title("Implied dividend yield"); plt.grid(alpha=0.3);

## Sensitivities of the forward (AAD)
One tape on the underlying's risk factors. Closed forms: $\partial F/\partial S = e^{-q_{repo}T}/P(0,T)$ (cash dividends
do not move with $S$), $\partial F/\partial q_{repo} = -T\,F$, $\partial F/\partial D_i = -P(0,pay_i)\,e^{-q_{repo}T}/P(0,T)$ for
$ex_i \le T$ (0 after).

In [ ]:
rf = eq.risk_factors()
with tf.GradientTape() as tape:
    F = eq.forward(expiry)
grads = tape.gradient(F, rf)

t = eq.year_fraction(expiry)
P = float(curve.discount(expiry))
print(f"dF/dS    = {float(grads[prefix + 'SPOT'][0]):.6f}   closed form {np.exp(-float(eq.repo) * t) / P:.6f}")
print(f"dF/drepo = {float(grads[prefix + 'REPO'][0]):.6f}   closed form {-t * float(F):.6f}")
for pillar, g in zip(curve.pillars, grads[curve.name]):
    print(f"dF/dr({pillar:>5.2f}Y) = {0.0 if g is None else float(g):10.4f}")
for ex, pay, amount, g in zip(eq.dividends.ex_dates, eq.dividends.payment_dates, eq.dividends.amounts,
                              grads[prefix + "DIV"][0].numpy()):
    closed = -float(curve.discount(pay)) * np.exp(-float(eq.repo) * t) / P if ex <= expiry else 0.0
    print(f"dF/dD({ex}, D={amount:7.4f}) = {g:9.6f}   closed form {closed:9.6f}")

Bumping the market variable moves the view: $\Delta F \approx \partial F/\partial S \cdot \Delta S$.

In [ ]:
spot_var = mkt.get_eq_spot(TICKER)
s0 = float(spot_var)
spot_var.assign(s0 + 1.0)
print(f"F(S0+1) - F(S0) = {float(eq.forward(expiry)) - float(F):.6f}")
spot_var.assign(s0);

## Exercises
1. Compare `eq.forward(expiry)` with the forward of `BlackScholesPricer` (`option.forward`) for both dividend models: why do they coincide?
2. Which pillar sensitivities of the forward are non-zero, and why only those?
3. Take a horizon between an ex-date and its payment date: is that dividend in the forward, and with which discount factor?